# 09 — Publicación: vistas para la aplicación y Power BI, y verificación

**Objetivo:** dejar todo listo para el consumo:
1. Crear las vistas de la Fase 9 (`sql/06_vistas_app.sql`) con permiso **solo para lectura**.
2. Verificar que la aplicación, conectada como `lector_bi`, **reproduce exactamente** las predicciones guardadas en la base (consistencia entre notebook y app).
3. Instrucciones para ejecutar la aplicación.

**Orden:** este notebook va después del 08. Si se vuelve a ejecutar el 04, hay que volver a ejecutar del 05 al 09.

## 1. Preparación

In [1]:
import sys
from pathlib import Path

RAIZ = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

import numpy as np
import pandas as pd
from sqlalchemy import text

from src.bd import obtener_motor, ejecutar_sql
from src.servicio import cargar_artefactos, leer_entradas, evaluar, explicar, alertas

## 2. Crear las vistas de la Fase 9

In [2]:
motor = obtener_motor()
ejecutar_sql(motor, '06_vistas_app.sql')

✅ Ejecutado sql/06_vistas_app.sql


## 3. ¿Qué puede ver el usuario de solo lectura?
📋 **Pega esta tabla.**

In [3]:
motor_bi = obtener_motor(rol='lector')
objetos = ['vw_cartera_cobranza', 'vw_resumen_cohortes', 'vw_entradas_modelo', 'vw_roi_campanas',
           'vw_roi_segmentos', 'segmentos', 'canales', 'campanas', 'campana_asignaciones',
           'creditos', 'perfil_solicitud', 'resultados_recuperacion', 'deudores']
filas = []
for o in objetos:
    try:
        with motor_bi.connect() as con:
            n = con.execute(text(f'SELECT COUNT(*) FROM {o}')).scalar()
        filas.append({'objeto': o, 'lector_bi': '✅ puede leer', 'filas': n})
    except Exception:
        filas.append({'objeto': o, 'lector_bi': '🔒 bloqueado', 'filas': None})
pd.DataFrame(filas)

,objeto,lector_bi,filas
0,vw_cartera_cobranza,✅ puede leer,17010.0
1,vw_resumen_cohortes,✅ puede leer,18.0
2,vw_entradas_modelo,✅ puede leer,16902.0
3,vw_roi_campanas,✅ puede leer,2.0
4,vw_roi_segmentos,✅ puede leer,10.0
5,segmentos,✅ puede leer,5.0
6,canales,✅ puede leer,5.0
7,campanas,✅ puede leer,2.0
8,campana_asignaciones,✅ puede leer,6326.0
9,creditos,🔒 bloqueado,NaN


## 4. Consistencia: ¿la app calcula lo mismo que quedó guardado?
Se toman 300 créditos al azar, se recalcula la predicción **como lo hace la aplicación** (vista de solo lectura + `src/servicio.py`) y se compara con la tabla `predicciones`.

📋 **Pega el resultado.**

In [4]:
art = cargar_artefactos()
entradas = leer_entradas(motor_bi)
muestra = entradas.sample(min(300, len(entradas)), random_state=42).reset_index(drop=True)
app = evaluar(muestra, art)
with motor.connect() as con:
    guardado = pd.read_sql(text('SELECT credito_id, prob_recuperar, segmento_id FROM predicciones WHERE es_vigente'), con)
guardado['prob_recuperar'] = guardado['prob_recuperar'].astype(float)
comp = app.merge(guardado, on='credito_id', suffixes=('_app', '_bd'))
dif = (comp['prob_recuperar_app'] - comp['prob_recuperar_bd']).abs().max()
mismo_seg = (comp['segmento_id_app'] == comp['segmento_id_bd']).mean()
print(f'Créditos comparados: {len(comp)}')
print(f'Diferencia máxima de probabilidad: {dif:.2e}  ' + ('✅' if dif < 1e-5 else '❌'))
print(f'Mismo segmento: {mismo_seg:.1%}  ' + ('✅' if mismo_seg == 1 else '❌'))

Créditos comparados: 300
Diferencia máxima de probabilidad: 5.00e-07  ✅
Mismo segmento: 100.0%  ✅


## 5. Ejemplo: explicación de un deudor de la cartera activa

In [5]:
with motor_bi.connect() as con:
    cid = con.execute(text("SELECT credito_id FROM vw_cartera_cobranza WHERE conjunto = 'inmadura' ORDER BY prioridad LIMIT 1")).scalar()
uno = leer_entradas(motor_bi, cid)
print(f'Crédito {cid} (prioridad 1 de la cartera activa)')
print(evaluar(uno, art).round(3).to_string(index=False))
for a in alertas(uno.iloc[0]):
    print(a)
explicar(uno, art)[['nombre', 'valor', 'contribucion']].round(3)

Crédito 10784 (prioridad 1 de la cartera activa)
 credito_id  prob_recuperar  saldo_imputado  saldo_estimado  monto_esperado  segmento_id
      10784           0.956           False         15000.0        4616.291            1
🚩 Default en la primera cuota: patrón típico de fraude. Verificar identidad antes de cobrar.


c:\Users\Cindy\Proyecto ll\cobranza-ml\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Cindy\Proyecto ll\cobranza-ml\.venv\Lib\site-packages\shap\explainers\_tree.py:578: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


,nombre,valor,contribucion
0,Cuota en la que cayó en default,1.000,2.972
1,Proporción del plazo pagada antes del default,0.028,0.819
2,% del crédito pendiente al default,1.000,0.215
3,Relación deuda / ingreso,0.350,0.069
4,Moras en los últimos 7 años,0.000,-0.068
5,Ingreso mensual,8.590,0.055
6,Consultas a la central de riesgo,3.000,0.051
7,Número de inversionistas,6.469,-0.039


## 6. Ejecutar la aplicación
En la terminal de VS Code (con `(.venv)` activo y dentro de `cobranza-ml`):

```powershell
streamlit run app/app.py
```

Se abre el navegador en `http://localhost:8501`. Para detenerla: `Ctrl + C` en la terminal.